In [1]:
!pip -q install nibabel pydicom simpleitk scikit-learn tqdm

# Improvement 2 — Lightweight Gated MRI-PET Fusion + Anti-Collapse Training + Reliability Analysis



## Main idea

The first improvement was too complex for the paired ADNI data. It used a cross-modal Transformer, prototype OOD analyzer, and FiLM harmonization, but it collapsed to majority-class prediction on the smaller dataset.

Improvement 2 is deliberately simpler and more grounded:

1. **Lightweight gated fusion** replaces the heavy cross-modal Transformer.
2. **Class-balanced focal loss** directly targets class collapse.
3. **Validation threshold tuning** avoids blindly using threshold 0.5.
4. **Post-hoc prototype reliability/OOD scoring** gives reliability evidence without modifying embeddings during training.

The model still uses MRI + PET and the same ADNI1 ↔ ADNI2 transfer setup.


In [2]:

import os
import gc
import math
import json
import time
import random
import zipfile
import warnings
from pathlib import Path
from itertools import cycle

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    f1_score,
    confusion_matrix,
)

import nibabel as nib
import pydicom
import SimpleITK as sitk

warnings.filterwarnings("ignore")

In [3]:



from pathlib import Path
import os
import random
import numpy as np
import torch

IN_COLAB = False
IN_KAGGLE = True

WORK_BASE = Path("/kaggle/working/improvement2_gated_reliability")
CACHE_ROOT = WORK_BASE / "cache_mmdda_fu"
CHECKPOINT_ROOT = WORK_BASE / "checkpoints_improvement2"

for p in [CACHE_ROOT, CHECKPOINT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

CSV_ROOT = Path("/kaggle/input/datasets/shayansarosh12/dl-proj-improvement2-csvs")
THREE_DATA_ROOT = Path("/kaggle/input/datasets/shayansarosh12/dl-proj")
ADNI1_MRI_DATA_ROOT = Path("/kaggle/input/datasets/syedmuabdullah99313/dl-project")

CSV_FILES = {
    "ADNI1_MRI": CSV_ROOT / "improvement2_adni1_mri_5_05_2026.csv",
    "ADNI1_PET": CSV_ROOT / "improvement2_adni1_pet_5_05_2026.csv",
    "ADNI2_MRI": CSV_ROOT / "improvement2_adni2_mri_5_05_2026.csv",
    "ADNI2_PET": CSV_ROOT / "improvement2_adni2_pet_5_05_2026.csv",
}

for k, path in CSV_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"{k} CSV not found at: {path}")


EXTRACTED = {
    "ADNI1_MRI": [ADNI1_MRI_DATA_ROOT / "ADNI"],
    "ADNI1_PET": [THREE_DATA_ROOT / "improvement2_adni1_pet_clean" / "ADNI"],
    "ADNI2_MRI": [THREE_DATA_ROOT / "improvement2_adni2_mri" / "ADNI"],
    "ADNI2_PET": [THREE_DATA_ROOT / "improvement2_adni2_pet_clean" / "ADNI"],
}

for key, roots in EXTRACTED.items():
    for root in roots:
        if not root.exists():
            raise FileNotFoundError(f"{key} image root does not exist: {root}")

print("CSV_ROOT:", CSV_ROOT)
print("THREE_DATA_ROOT:", THREE_DATA_ROOT)
print("ADNI1_MRI_DATA_ROOT:", ADNI1_MRI_DATA_ROOT)

print("\nCSV files:")
for k, v in CSV_FILES.items():
    print(k, "->", v)

print("\nImage roots:")
for k, v in EXTRACTED.items():
    print(k, "->", v)

SEED = 42
TARGET_SHAPE = (113, 137, 113)
MAX_PAIR_GAP_DAYS = 180
TASK_NAME = "AD_vs_CN"

BATCH_SIZE = 1
LR = 9e-7
DROPOUT = 0.5
PRETRAIN_EPOCHS = 150
ADAPT_EPOCHS = 50
GRL_LAMBDA = 0.1

FOCAL_GAMMA = 2.0
MODALITY_ALIGN_WEIGHT = 0.05
DOMAIN_LOSS_WEIGHT = 0.10

CHECKPOINT_MONITOR = "AUC"
THRESHOLD_METRIC = "balanced_accuracy"  

PREFILTER_BEFORE_SPLIT = True
CLEAR_CACHE = False  
PREFER_PREPROCESSED_CANDIDATES = True
RAW_APPROX_FOREGROUND_CROP = True
RAW_APPROX_GAUSSIAN_SIGMA = 1.0
RAW_APPROX_USE_N4_BIAS_CORRECTION = False

PAPER_AD_CN_TOTAL = 279 + 180
MIN_RELIABLE_SOURCE_SIZE = 50

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nWORK_BASE:", WORK_BASE)
print("CACHE_ROOT:", CACHE_ROOT)
print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)
print("DEVICE:", DEVICE)


CSV_ROOT: /kaggle/input/datasets/shayansarosh12/dl-proj-improvement2-csvs
THREE_DATA_ROOT: /kaggle/input/datasets/shayansarosh12/dl-proj
ADNI1_MRI_DATA_ROOT: /kaggle/input/datasets/syedmuabdullah99313/dl-project

CSV files:
ADNI1_MRI -> /kaggle/input/datasets/shayansarosh12/dl-proj-improvement2-csvs/improvement2_adni1_mri_5_05_2026.csv
ADNI1_PET -> /kaggle/input/datasets/shayansarosh12/dl-proj-improvement2-csvs/improvement2_adni1_pet_5_05_2026.csv
ADNI2_MRI -> /kaggle/input/datasets/shayansarosh12/dl-proj-improvement2-csvs/improvement2_adni2_mri_5_05_2026.csv
ADNI2_PET -> /kaggle/input/datasets/shayansarosh12/dl-proj-improvement2-csvs/improvement2_adni2_pet_5_05_2026.csv

Image roots:
ADNI1_MRI -> [PosixPath('/kaggle/input/datasets/syedmuabdullah99313/dl-project/ADNI')]
ADNI1_PET -> [PosixPath('/kaggle/input/datasets/shayansarosh12/dl-proj/improvement2_adni1_pet_clean/ADNI')]
ADNI2_MRI -> [PosixPath('/kaggle/input/datasets/shayansarosh12/dl-proj/improvement2_adni2_mri/ADNI')]
ADNI2_PET

## 1. Sanity checks for mounted Kaggle datasets



In [4]:

print("\nDetailed folder check:")
for key, roots in EXTRACTED.items():
    print("\n" + key)
    root = roots[0]
    print("  selected root:", root)
    children = list(root.iterdir())
    dirs = [x for x in children if x.is_dir()]
    files = [x for x in children if x.is_file()]
    print("  child dirs:", len(dirs))
    print("  child files:", len(files))
    print("  first dirs:", [d.name for d in dirs[:10]])
    print("  first files:", [f.name for f in files[:10]])



Detailed folder check:

ADNI1_MRI
  selected root: /kaggle/input/datasets/syedmuabdullah99313/dl-project/ADNI
  child dirs: 396
  child files: 0
  first dirs: ['133_S_0913', '094_S_1090', '128_S_0272', '027_S_1387', '031_S_0618', '027_S_1277', '137_S_0669', '005_S_0221', '002_S_0685', '094_S_0531']
  first files: []

ADNI1_PET
  selected root: /kaggle/input/datasets/shayansarosh12/dl-proj/improvement2_adni1_pet_clean/ADNI
  child dirs: 76
  child files: 0
  first dirs: ['128_S_0272', '031_S_0618', '137_S_0841', '131_S_0497', '007_S_1339', '029_S_0845', '033_S_1283', '024_S_1171', '006_S_0547', '130_S_0232']
  first files: []

ADNI2_MRI
  selected root: /kaggle/input/datasets/shayansarosh12/dl-proj/improvement2_adni2_mri/ADNI
  child dirs: 399
  child files: 0
  first dirs: ['082_S_5029', '067_S_0257', '022_S_4266', '128_S_4599', '037_S_5126', '002_S_0685', '032_S_0214', '116_S_4092', '011_S_4105', '136_S_4269']
  first files: []

ADNI2_PET
  selected root: /kaggle/input/datasets/shaya

In [5]:

for key, path in CSV_FILES.items():
    df = pd.read_csv(path)
    print("\n", key)
    print("path:", path)
    print("rows:", len(df))
    print("unique subjects:", df["Subject"].nunique())
    print(df["Group"].value_counts())



 ADNI1_MRI
path: /kaggle/input/datasets/shayansarosh12/dl-proj-improvement2-csvs/improvement2_adni1_mri_5_05_2026.csv
rows: 1862
unique subjects: 400
Group
MCI    1005
CN      535
AD      322
Name: count, dtype: int64

 ADNI1_PET
path: /kaggle/input/datasets/shayansarosh12/dl-proj-improvement2-csvs/improvement2_adni1_pet_5_05_2026.csv
rows: 394
unique subjects: 85
Group
CN    226
AD    168
Name: count, dtype: int64

 ADNI2_MRI
path: /kaggle/input/datasets/shayansarosh12/dl-proj-improvement2-csvs/improvement2_adni2_mri_5_05_2026.csv
rows: 1547
unique subjects: 412
Group
CN     905
AD     344
MCI    298
Name: count, dtype: int64

 ADNI2_PET
path: /kaggle/input/datasets/shayansarosh12/dl-proj-improvement2-csvs/improvement2_adni2_pet_5_05_2026.csv
rows: 747
unique subjects: 549
Group
CN     428
AD     172
MCI    147
Name: count, dtype: int64


In [6]:

adni1_mri_csv = pd.read_csv(CSV_FILES["ADNI1_MRI"])
adni1_mri_root = EXTRACTED["ADNI1_MRI"][0]

csv_subjects = set(adni1_mri_csv["Subject"].astype(str))
folder_subjects = set([p.name for p in adni1_mri_root.iterdir() if p.is_dir()])

print("ADNI1 MRI CSV unique subjects:", len(csv_subjects))
print("ADNI1 MRI folder subject dirs:", len(folder_subjects))
print("Subjects in both:", len(csv_subjects & folder_subjects))
print("CSV subjects missing from folders:", len(csv_subjects - folder_subjects))
print("Folder subjects not in CSV:", len(folder_subjects - csv_subjects))
print("\nExample matched subjects:")
print(sorted(list(csv_subjects & folder_subjects))[:20])


ADNI1 MRI CSV unique subjects: 400
ADNI1 MRI folder subject dirs: 396
Subjects in both: 396
CSV subjects missing from folders: 4
Folder subjects not in CSV: 0

Example matched subjects:
['002_S_0295', '002_S_0413', '002_S_0559', '002_S_0619', '002_S_0685', '002_S_0729', '002_S_0782', '002_S_0816', '002_S_0938', '002_S_0954', '002_S_0955', '002_S_1018', '002_S_1070', '002_S_1155', '002_S_1261', '002_S_1268', '002_S_1280', '005_S_0221', '005_S_0222', '005_S_0223']


In [7]:

SANITY_MODE = False

if SANITY_MODE:
    PRETRAIN_EPOCHS = 2
    ADAPT_EPOCHS = 2
    print("SANITY_MODE is ON -> using tiny epoch counts for debugging.")
else:
    print("SANITY_MODE is OFF -> using full epoch counts.")


SANITY_MODE is OFF -> using full epoch counts.


## 2. Load CSVs and build AD/CN MRI-PET pairs

In [8]:
def load_csv(path: Path, modality_name: str, dataset_name: str):
    if not path.exists():
        raise FileNotFoundError(f"Missing CSV: {path}")

    df = pd.read_csv(path).copy()
    rename_map = {
        "Image Data ID": "image_id",
        "Subject": "subject",
        "Group": "group",
        "Sex": "sex",
        "Age": "age",
        "Visit": "visit",
        "Modality": "modality",
        "Description": "description",
        "Type": "type",
        "Acq Date": "acq_date",
        "Format": "format",
        "Downloaded": "downloaded",
    }
    df = df.rename(columns=rename_map)
    df["dataset"] = dataset_name
    df["modality_name"] = modality_name
    df["acq_date"] = pd.to_datetime(df["acq_date"], errors="coerce")
    df["subject"] = df["subject"].astype(str)
    df["visit"] = df["visit"].astype(str)
    df["group"] = df["group"].astype(str).str.upper()
    df["label_num"] = df["group"].map({"CN": 0, "AD": 1})
    df = df[df["group"].isin(["CN", "AD"])].copy()
    return df

adni1_mri = load_csv(CSV_FILES["ADNI1_MRI"], "MRI", "ADNI1")
adni1_pet = load_csv(CSV_FILES["ADNI1_PET"], "PET", "ADNI1")
adni2_mri = load_csv(CSV_FILES["ADNI2_MRI"], "MRI", "ADNI2")
adni2_pet = load_csv(CSV_FILES["ADNI2_PET"], "PET", "ADNI2")

adni1_pet = adni1_pet[
    ~adni1_pet["description"]
    .astype(str)
    .str.upper()
    .str.contains("PIB|C11|C-11", regex=True, na=False)
].copy()

print("ADNI1 PET after removing PIB/C11 rows:", adni1_pet.shape)
print("ADNI1 PET group counts:", adni1_pet["group"].value_counts().to_dict())
print("ADNI1 PET unique subjects:", adni1_pet["subject"].nunique())


for name, df in [
    ("ADNI1 MRI", adni1_mri),
    ("ADNI1 PET", adni1_pet),
    ("ADNI2 MRI", adni2_mri),
    ("ADNI2 PET", adni2_pet),
]:
    print(f"\n{name}: {df.shape}")
    print(df[['subject', 'group', 'visit', 'image_id', 'acq_date']].head(3))
    print(df['group'].value_counts().to_dict())

ADNI1 PET after removing PIB/C11 rows: (339, 15)
ADNI1 PET group counts: {'CN': 197, 'AD': 142}
ADNI1 PET unique subjects: 85

ADNI1 MRI: (857, 15)
       subject group visit image_id   acq_date
8   137_S_1041    AD    sc   I29268 2006-11-09
9   137_S_1041    AD   m06   I56100 2007-06-06
10  137_S_1041    AD   m12   I84667 2007-12-12
{'CN': 535, 'AD': 322}

ADNI1 PET: (339, 15)
      subject group visit image_id   acq_date
0  137_S_1041    AD    bl   I32968 2006-12-12
1  137_S_1041    AD   m06   I55363 2007-05-29
2  137_S_1041    AD   m12   I85759 2007-12-18
{'CN': 197, 'AD': 142}

ADNI2 MRI: (1249, 15)
      subject group visit image_id   acq_date
0  941_S_4376    CN   v02  I269043 2011-06-01
1  941_S_4376    CN   v02  I276860 2012-01-06
2  941_S_4376    CN   v04  I294442 2012-03-30
{'CN': 905, 'AD': 344}

ADNI2 PET: (600, 15)
      subject group visit image_id   acq_date
0  941_S_4376    CN   v03  I283469 2012-02-08
1  941_S_4365    CN   v03  I274743 2011-12-30
2  941_S_4365    CN   

In [9]:
def build_nearest_pairs(mri_df, pet_df, dataset_name, max_gap_days=180):
    shared_subjects = sorted(set(mri_df["subject"]) & set(pet_df["subject"]))
    rows = []

    for subj in shared_subjects:
        msub = mri_df[mri_df["subject"] == subj].copy()
        psub = pet_df[pet_df["subject"] == subj].copy()

        best = None
        best_key = None

        for _, mr in msub.iterrows():
            for _, pr in psub.iterrows():
                same_group = int(mr["group"] == pr["group"])
                if pd.isna(mr["acq_date"]) or pd.isna(pr["acq_date"]):
                    gap_days = 999999
                else:
                    gap_days = abs((mr["acq_date"] - pr["acq_date"]).days)


                score = (-same_group, gap_days)

                if best is None or score < best_key:
                    best = (mr, pr)
                    best_key = score

        if best is None:
            continue

        mr, pr = best
        if mr["group"] != pr["group"]:
            continue

        if pd.isna(mr["acq_date"]) or pd.isna(pr["acq_date"]):
            continue

        gap_days = abs((mr["acq_date"] - pr["acq_date"]).days)
        if gap_days > max_gap_days:
            continue

        rows.append({
            "dataset": dataset_name,
            "subject": subj,
            "group": mr["group"],
            "label_num": int(mr["label_num"]),
            "sex": mr["sex"],
            "age": float(mr["age"]),
            "mri_visit": mr["visit"],
            "pet_visit": pr["visit"],
            "mri_image_id": str(mr["image_id"]),
            "pet_image_id": str(pr["image_id"]),
            "mri_date": mr["acq_date"],
            "pet_date": pr["acq_date"],
            "pair_gap_days": int(gap_days),
        })

    out = pd.DataFrame(rows).sort_values(["dataset", "subject"]).reset_index(drop=True)
    return out

paired_adni1 = build_nearest_pairs(adni1_mri, adni1_pet, "ADNI1", max_gap_days=MAX_PAIR_GAP_DAYS)
paired_adni2 = build_nearest_pairs(adni2_mri, adni2_pet, "ADNI2", max_gap_days=MAX_PAIR_GAP_DAYS)

print("paired_adni1:", paired_adni1.shape)
print(paired_adni1["group"].value_counts().to_dict())
print(paired_adni1["pair_gap_days"].describe())

print("\npaired_adni2:", paired_adni2.shape)
print(paired_adni2["group"].value_counts().to_dict())
print(paired_adni2["pair_gap_days"].describe())

display(paired_adni1.head())
display(paired_adni2.head())

paired_adni1: (84, 13)
{'CN': 43, 'AD': 41}
count    84.000000
mean      7.392857
std      13.908494
min       0.000000
25%       0.000000
50%       0.500000
75%       7.250000
max      65.000000
Name: pair_gap_days, dtype: float64

paired_adni2: (281, 13)
{'CN': 181, 'AD': 100}
count    281.000000
mean      19.843416
std       24.948168
min        0.000000
25%        4.000000
50%       13.000000
75%       26.000000
max      159.000000
Name: pair_gap_days, dtype: float64


,dataset,subject,group,label_num,sex,age,mri_visit,pet_visit,mri_image_id,pet_image_id,mri_date,pet_date,pair_gap_days
0,ADNI1,005_S_0221,AD,1,M,68.0,m06,m06,I25942,I25918,2006-10-06,2006-10-06,0
1,ADNI1,005_S_0223,CN,0,F,79.0,m06,m06,I26116,I26108,2006-10-11,2006-10-11,0
2,ADNI1,005_S_0610,CN,0,M,80.0,m06,m06,I39068,I39087,2007-02-09,2007-02-09,0
3,ADNI1,005_S_0929,AD,1,M,83.0,m06,m06,I53858,I57389,2007-05-09,2007-06-19,41
4,ADNI1,005_S_1341,AD,1,F,72.0,m06,m06,I76567,I76770,2007-10-02,2007-10-02,0


,dataset,subject,group,label_num,sex,age,mri_visit,pet_visit,mri_image_id,pet_image_id,mri_date,pet_date,pair_gap_days
0,ADNI2,002_S_0295,CN,0,M,90.0,v06,v06,I238627,I239487,2011-06-02,2011-06-09,7
1,ADNI2,002_S_0413,CN,0,F,82.0,v06,v06,I240812,I240813,2011-06-16,2011-06-17,1
2,ADNI2,002_S_0685,CN,0,F,96.0,v11,v11,I322437,I321228,2012-07-27,2012-08-02,6
3,ADNI2,002_S_1261,CN,0,F,77.0,v11,v11,I361610,I363184,2013-02-27,2013-03-07,8
4,ADNI2,002_S_1280,CN,0,F,77.0,v11,v11,I361326,I360640,2013-02-26,2013-02-19,7


## 3. Build file path indexes



In [10]:
from pathlib import Path
from tqdm.auto import tqdm

print("Building path indexes using selected ADNI roots only. This may take time but avoids duplicate parent-folder scans.")

VOLUME_EXTENSIONS = (
    ".nii", ".nii.gz", ".img", ".hdr", ".v", ".mnc", ".nrrd", ".mha", ".mhd", ".mgz"
)

def build_path_index(search_roots):
    index = {
        "dirs": [],
        "vol_files": [],
    }

    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue

        print(f"Scanning root: {root}")

        count = 0
        for p in root.rglob("*"):
            count += 1
            if count % 50000 == 0:
                print(f"  scanned {count:,} paths... dirs={len(index['dirs']):,}, vol_files={len(index['vol_files']):,}")

            low = str(p).lower()
            if p.is_dir():
                index["dirs"].append((low, p))
            elif p.is_file() and low.endswith(VOLUME_EXTENSIONS):
                index["vol_files"].append((low, p))

        print(f"Finished root: {root}")
        print(f"  total scanned: {count:,}")
        print(f"  dirs: {len(index['dirs']):,}")
        print(f"  volume files: {len(index['vol_files']):,}")

    return index

INDEXES = {
    "ADNI1_MRI": build_path_index(EXTRACTED["ADNI1_MRI"]),
    "ADNI1_PET": build_path_index(EXTRACTED["ADNI1_PET"]),
    "ADNI2_MRI": build_path_index(EXTRACTED["ADNI2_MRI"]),
    "ADNI2_PET": build_path_index(EXTRACTED["ADNI2_PET"]),
}

for k, idx in INDEXES.items():
    print(k, "dirs:", len(idx["dirs"]), "| vol files:", len(idx["vol_files"]))

Building path indexes using selected ADNI roots only. This may take time but avoids duplicate parent-folder scans.
Scanning root: /kaggle/input/datasets/syedmuabdullah99313/dl-project/ADNI
  scanned 50,000 paths... dirs=1,370, vol_files=0
  scanned 100,000 paths... dirs=1,950, vol_files=0
  scanned 150,000 paths... dirs=2,550, vol_files=0
  scanned 200,000 paths... dirs=3,114, vol_files=0
  scanned 250,000 paths... dirs=3,690, vol_files=0
Finished root: /kaggle/input/datasets/syedmuabdullah99313/dl-project/ADNI
  total scanned: 273,845
  dirs: 3,965
  volume files: 0
Scanning root: /kaggle/input/datasets/shayansarosh12/dl-proj/improvement2_adni1_pet_clean/ADNI
Finished root: /kaggle/input/datasets/shayansarosh12/dl-proj/improvement2_adni1_pet_clean/ADNI
  total scanned: 43,762
  dirs: 842
  volume files: 503
Scanning root: /kaggle/input/datasets/shayansarosh12/dl-proj/improvement2_adni2_mri/ADNI
  scanned 50,000 paths... dirs=1,385, vol_files=0
  scanned 100,000 paths... dirs=1,949, vo

## 4. Volume loading, preprocessing, caching, and readability filtering



In [11]:

import shutil

if CLEAR_CACHE:
    shutil.rmtree(CACHE_ROOT, ignore_errors=True)
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Cache cleared.")
else:
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    print("CLEAR_CACHE=False -> keeping/reusing existing cache if present.")
    print("Existing cached npy files:", len(list(CACHE_ROOT.glob("*.npy"))))


CLEAR_CACHE=False -> keeping/reusing existing cache if present.
Existing cached npy files: 0


In [12]:
def locate_study_path(index, image_id, subject, visit=None):
    image_id = str(image_id).lower()
    subject = str(subject).lower()
    visit = None if visit is None else str(visit).lower()

    def is_v_file(p):
        return str(p).lower().endswith(".v")

    hit_dirs = [p for low, p in index["dirs"] if image_id in low]
    if hit_dirs:
        return sorted(hit_dirs, key=lambda x: len(str(x)), reverse=True)[0]

    hit_files = [p for low, p in index["vol_files"] if image_id in low and not is_v_file(p)]
    if hit_files:
        return sorted(hit_files, key=lambda x: x.stat().st_size if x.exists() else 0, reverse=True)[0]

    hit_v_files = [p for low, p in index["vol_files"] if image_id in low and is_v_file(p)]
    if hit_v_files:
        return sorted(hit_v_files, key=lambda x: x.stat().st_size if x.exists() else 0, reverse=True)[0]

    subject_dirs = []
    for low, p in index["dirs"]:
        if subject in low and (visit is None or visit in low):
            subject_dirs.append(p)

    if subject_dirs:
        return sorted(subject_dirs, key=lambda x: len(str(x)), reverse=True)[0]

    subject_dirs = [p for low, p in index["dirs"] if subject in low]
    if subject_dirs:
        return sorted(subject_dirs, key=lambda x: len(str(x)), reverse=True)[0]


    subject_files = []
    for low, p in index["vol_files"]:
        if subject in low and (visit is None or visit in low) and not is_v_file(p):
            subject_files.append(p)

    if subject_files:
        return sorted(subject_files, key=lambda x: x.stat().st_size if x.exists() else 0, reverse=True)[0]


    subject_v_files = []
    for low, p in index["vol_files"]:
        if subject in low and (visit is None or visit in low) and is_v_file(p):
            subject_v_files.append(p)

    if subject_v_files:
        return sorted(subject_v_files, key=lambda x: x.stat().st_size if x.exists() else 0, reverse=True)[0]

    raise FileNotFoundError(
        f"Could not locate study path for image_id={image_id}, subject={subject}, visit={visit}"
    )

In [13]:
PAPER_PREPROCESSED_KEYWORDS = (
    "cat12", "spm", "mni", "smwc", "mwp", "warped", "normalized", "normalised",
    "registered", "coreg", "brain", "strip", "skull", "resliced", "resample",
    "smooth", "wc1", "wc2", "mrireg", "petreg"
)

def _is_probably_preprocessed_path(path: Path):
    low = str(path).lower()
    return any(k in low for k in PAPER_PREPROCESSED_KEYWORDS)


def _ensure_3d_volume(arr, source):
    arr = np.asarray(arr, dtype=np.float32)
    while arr.ndim > 3:
        arr = arr[..., 0]
    if arr.ndim != 3:
        raise RuntimeError(f"Expected 3D volume from {source}, got shape {arr.shape}")
    return arr.astype(np.float32)


def _load_analyze_like_hdr_pair(hdr_path: Path):
    hdr_path = Path(hdr_path)
    low = str(hdr_path).lower()

    if not low.endswith(".hdr"):
        raise RuntimeError(f"Not a .hdr file: {hdr_path}")

    candidates = [hdr_path.with_suffix(".img")]

    if low.endswith(".i.hdr"):
        candidates.append(Path(str(hdr_path)[:-4]))

    candidates.append(Path(str(hdr_path)[:-4]))

    seen = set()
    unique_candidates = []
    for c in candidates:
        s = str(c)
        if s not in seen:
            seen.add(s)
            unique_candidates.append(c)

    data_path = None
    for c in unique_candidates:
        if c.exists() and c.is_file():
            data_path = c
            break

    if data_path is None:
        raise RuntimeError(f"Could not find paired binary file for header: {hdr_path}")

    try:
        from nibabel.analyze import AnalyzeHeader

        with open(hdr_path, "rb") as f_hdr:
            hdr = AnalyzeHeader.from_fileobj(f_hdr)

        with open(data_path, "rb") as f_img:
            arr = hdr.data_from_fileobj(f_img)

        return _ensure_3d_volume(arr, f"{hdr_path} + {data_path}")
    except Exception as e:
        raise RuntimeError(f"Failed to load Analyze-like pair: {hdr_path} + {data_path}") from e


def load_single_volume_file(path: Path):
    path = Path(path)
    low = str(path).lower()

    if low.endswith(".v"):
        try:
            from nibabel import ecat
            img = ecat.load(str(path))
            arr = np.asarray(img.get_fdata(), dtype=np.float32)
            return _ensure_3d_volume(arr, path)
        except Exception:
            pass

    if low.endswith(".hdr"):
        try:
            return _load_analyze_like_hdr_pair(path)
        except Exception:
            pass

    try:
        img = nib.load(str(path))
        arr = np.asarray(img.get_fdata(), dtype=np.float32)
        return _ensure_3d_volume(arr, path)
    except Exception:
        pass

    try:
        img = sitk.ReadImage(str(path))
        arr = sitk.GetArrayFromImage(img).astype(np.float32)
        arr = np.transpose(arr, (2, 1, 0))
        return _ensure_3d_volume(arr, path)
    except Exception as e:
        raise RuntimeError(f"Could not load single-volume file: {path}") from e


def _dicom_sort_value(ds):
    instance_no = getattr(ds, "InstanceNumber", None)
    if instance_no is not None:
        try:
            return float(instance_no)
        except Exception:
            pass

    ipp = getattr(ds, "ImagePositionPatient", None)
    if ipp is not None and len(ipp) >= 3:
        try:
            return float(ipp[-1])
        except Exception:
            pass

    slice_location = getattr(ds, "SliceLocation", None)
    if slice_location is not None:
        try:
            return float(slice_location)
        except Exception:
            pass

    return 0.0


def load_dicom_series_from_dir(series_dir: Path):
    series_dir = Path(series_dir)

    try:
        reader = sitk.ImageSeriesReader()
        series_ids = reader.GetGDCMSeriesIDs(str(series_dir))
        if series_ids:
            series_candidates = []
            for sid in series_ids:
                files = list(reader.GetGDCMSeriesFileNames(str(series_dir), sid))
                series_candidates.append((len(files), sid, files))

            for _, sid, files in sorted(series_candidates, reverse=True):
                try:
                    reader.SetFileNames(files)
                    img = reader.Execute()
                    arr = sitk.GetArrayFromImage(img).astype(np.float32)
                    arr = np.transpose(arr, (2, 1, 0))
                    return _ensure_3d_volume(arr, f"{series_dir}::{sid}")
                except Exception:
                    continue
    except Exception:
        pass

    grouped_slices = {}
    multiframe_volumes = []

    for p in series_dir.rglob("*"):
        if not p.is_file():
            continue
        try:
            ds = pydicom.dcmread(str(p), stop_before_pixels=False, force=True)
            if not hasattr(ds, "pixel_array"):
                continue
            arr = np.asarray(ds.pixel_array, dtype=np.float32)
        except Exception:
            continue

        series_uid = str(getattr(ds, "SeriesInstanceUID", "unknown"))

        if arr.ndim == 3:
            vol = np.transpose(arr, (1, 2, 0))
            multiframe_volumes.append((vol.size, vol))
            continue

        if arr.ndim != 2:
            continue

        rows = int(getattr(ds, "Rows", arr.shape[0]))
        cols = int(getattr(ds, "Columns", arr.shape[1]))
        key = (series_uid, rows, cols)
        grouped_slices.setdefault(key, []).append((_dicom_sort_value(ds), arr))

    volume_candidates = []

    for _, vol in multiframe_volumes:
        try:
            volume_candidates.append((vol.size, _ensure_3d_volume(vol, series_dir)))
        except Exception:
            continue

    for key, slices in grouped_slices.items():
        shapes = {sl.shape for _, sl in slices}
        if len(shapes) != 1 or len(slices) == 0:
            continue
        slices = sorted(slices, key=lambda x: x[0])
        try:
            vol = np.stack([x[1] for x in slices], axis=-1).astype(np.float32)
            volume_candidates.append((vol.size, _ensure_3d_volume(vol, f"{series_dir}::{key[0]}")))
        except Exception:
            continue

    if not volume_candidates:
        raise RuntimeError(f"No readable DICOM slices found inside {series_dir}")

    volume_candidates = sorted(volume_candidates, key=lambda x: x[0], reverse=True)
    return volume_candidates[0][1]


def _rank_volume_candidate(p: Path):
    low = str(p).lower()
    size = p.stat().st_size if p.exists() else 0

    preferred_bonus = -20 if (PREFER_PREPROCESSED_CANDIDATES and _is_probably_preprocessed_path(p)) else 0

    if low.endswith(".nii.gz"):
        rank = 0
    elif low.endswith(".nii"):
        rank = 1
    elif low.endswith(".mgz"):
        rank = 2
    elif low.endswith(".mha") or low.endswith(".mhd"):
        rank = 3
    elif low.endswith(".img"):
        rank = 4
    elif low.endswith(".hdr"):
        rank = 5
    elif low.endswith(".mnc"):
        rank = 6
    elif low.endswith(".nrrd"):
        rank = 7
    elif low.endswith(".v"):
        rank = 99
    else:
        rank = 50

    return (preferred_bonus + rank, -size)


def load_volume(path: Path):
    path = Path(path)

    if path.is_file():
        return load_single_volume_file(path)

    try:
        return load_dicom_series_from_dir(path)
    except Exception:
        pass

    volume_files = []
    for p in path.rglob("*"):
        if p.is_file():
            low = str(p).lower()
            if low.endswith(VOLUME_EXTENSIONS):
                volume_files.append(p)

    if not volume_files:
        raise RuntimeError(f"No readable volume found inside directory: {path}")

    volume_files = sorted(volume_files, key=_rank_volume_candidate)

    last_error = None
    for vf in volume_files:
        try:
            return load_single_volume_file(vf)
        except Exception as e:
            last_error = e
            continue

    raise RuntimeError(f"All candidate volume loaders failed for {path}") from last_error


In [14]:
def _ensure_3d_float(volume):
    v = np.asarray(volume, dtype=np.float32)
    v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)

    while v.ndim > 3:
        v = v[..., 0]

    if v.ndim != 3:
        raise ValueError(f"Expected 3D volume after squeezing, got shape {v.shape}")

    return v.astype(np.float32)


def robust_clip(volume, lower_q=0.5, upper_q=99.5):
    v = np.asarray(volume, dtype=np.float32)
    finite_mask = np.isfinite(v)
    if finite_mask.sum() == 0:
        return np.zeros_like(v, dtype=np.float32)
    lo, hi = np.percentile(v[finite_mask], [lower_q, upper_q])
    v = np.clip(v, lo, hi)
    return v.astype(np.float32)


def crop_to_foreground(volume, modality, margin=4):
    v = _ensure_3d_float(volume)
    nonzero = np.abs(v) > 0
    if nonzero.any():
        foreground_values = v[nonzero]
    else:
        foreground_values = v[np.isfinite(v)]

    if foreground_values.size == 0:
        return v

    if modality.upper() == "MRI":
        threshold = np.percentile(foreground_values, 35)
    else:
        threshold = np.percentile(foreground_values, 55)

    mask = (v > threshold) | nonzero
    coords = np.argwhere(mask)
    if coords.size == 0:
        return v

    lo = np.maximum(coords.min(axis=0) - margin, 0)
    hi = np.minimum(coords.max(axis=0) + margin + 1, np.array(v.shape))
    cropped = v[
        int(lo[0]):int(hi[0]),
        int(lo[1]):int(hi[1]),
        int(lo[2]):int(hi[2]),
    ]
    return cropped.astype(np.float32) if cropped.size > 0 else v


def maybe_n4_bias_correct(volume):
    v = _ensure_3d_float(volume)
    if not RAW_APPROX_USE_N4_BIAS_CORRECTION:
        return v

    try:
        itk = sitk.GetImageFromArray(np.transpose(v, (2, 1, 0)))
        mask = sitk.OtsuThreshold(itk, 0, 1, 128)
        corrected = sitk.N4BiasFieldCorrection(itk, mask)
        out = sitk.GetArrayFromImage(corrected).astype(np.float32)
        return np.transpose(out, (2, 1, 0))
    except Exception:
        return v


def maybe_gaussian_smooth(volume, sigma):
    v = _ensure_3d_float(volume)
    if sigma is None or sigma <= 0:
        return v

    try:
        itk = sitk.GetImageFromArray(np.transpose(v, (2, 1, 0)))
        smoothed = sitk.SmoothingRecursiveGaussian(itk, sigma=float(sigma))
        out = sitk.GetArrayFromImage(smoothed).astype(np.float32)
        return np.transpose(out, (2, 1, 0))
    except Exception:
        return v


def normalize_mri(volume):
    v = robust_clip(volume)
    mask = v > np.percentile(v, 20)
    if mask.sum() == 0:
        mean = v.mean()
        std = v.std() + 1e-6
    else:
        mean = v[mask].mean()
        std = v[mask].std() + 1e-6
    v = (v - mean) / std
    return v.astype(np.float32)


def normalize_pet(volume):
    v = robust_clip(volume)
    v = v - v.min()
    denom = v.max() + 1e-6
    v = v / denom
    return v.astype(np.float32)


def resize_volume(volume, target_shape=TARGET_SHAPE):
    x = torch.tensor(volume, dtype=torch.float32).unsqueeze(0).unsqueeze(0) 
    x = F.interpolate(x, size=target_shape, mode="trilinear", align_corners=False)
    return x.squeeze(0).squeeze(0).cpu().numpy().astype(np.float32)


def preprocess_volume(volume, modality, source_path=None):
    v = _ensure_3d_float(volume)
    source_path = None if source_path is None else Path(source_path)
    paper_like_input = (
        source_path is not None
        and PREFER_PREPROCESSED_CANDIDATES
        and _is_probably_preprocessed_path(source_path)
    )

    if not paper_like_input and RAW_APPROX_FOREGROUND_CROP:
        v = crop_to_foreground(v, modality=modality, margin=4)

    if modality.upper() == "MRI":
        if not paper_like_input:
            v = maybe_n4_bias_correct(v)
            v = maybe_gaussian_smooth(v, sigma=RAW_APPROX_GAUSSIAN_SIGMA)
        v = normalize_mri(v)
    elif modality.upper() == "PET":
        if not paper_like_input:
            v = maybe_gaussian_smooth(v, sigma=RAW_APPROX_GAUSSIAN_SIGMA)
        v = normalize_pet(v)
    else:
        raise ValueError(f"Unknown modality: {modality}")

    if tuple(v.shape) != tuple(TARGET_SHAPE):
        v = resize_volume(v, target_shape=TARGET_SHAPE)

    if modality.upper() == "MRI":
        v = normalize_mri(v)
    else:
        v = normalize_pet(v)

    return v.astype(np.float32)


In [15]:
def cache_key(row, modality):
    image_id = row["mri_image_id"] if modality == "MRI" else row["pet_image_id"]
    return f"{row['dataset']}__{row['subject']}__{modality}__{image_id}.npy"


def get_cached_or_build(row, modality, index_key):
    out_path = CACHE_ROOT / cache_key(row, modality)

    if out_path.exists():
        try:
            arr = np.load(out_path)
            return arr.astype(np.float32)
        except Exception:
            try:
                out_path.unlink()
            except Exception:
                pass

    if modality == "MRI":
        image_id = row["mri_image_id"]
        visit = row["mri_visit"]
    else:
        image_id = row["pet_image_id"]
        visit = row["pet_visit"]

    index = INDEXES[index_key]

    try:
        source_path = locate_study_path(
            index,
            image_id=image_id,
            subject=row["subject"],
            visit=visit,
        )
        volume = load_volume(source_path)
        volume = preprocess_volume(volume, modality=modality, source_path=source_path)
        volume = volume.astype(np.float32)
        np.save(out_path, volume)
        return volume
    except Exception:
        try:
            if out_path.exists():
                out_path.unlink()
        except Exception:
            pass
        raise


In [16]:
import signal
from tqdm.auto import tqdm
import pandas as pd

class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException("Image loading timed out")


def filter_loadable_pairs(df, name="dataset", timeout_sec=60):
    """
    Filters readable MRI-PET pairs using the notebook's real loading pipeline.
    It does NOT look for mri_path/pet_path columns.
    
    It uses:
      dataset
      mri_image_id
      pet_image_id
      get_cached_or_build(...)
    
    Returns:
      kept_df, bad_df
    """

    df = df.reset_index(drop=True).copy()

    kept_rows = []
    bad_rows = []

    signal.signal(signal.SIGALRM, timeout_handler)

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"filter {name}"):

        dataset = row["dataset"]

        mri_index_key = f"{dataset}_MRI"
        pet_index_key = f"{dataset}_PET"

        try:
            signal.alarm(timeout_sec)

            _ = get_cached_or_build(row, modality="MRI", index_key=mri_index_key)

            signal.alarm(timeout_sec)

            _ = get_cached_or_build(row, modality="PET", index_key=pet_index_key)

            signal.alarm(0)

            kept_rows.append(row)

        except TimeoutException:
            signal.alarm(0)
            bad_row = row.copy()
            bad_row["drop_reason"] = "timeout"
            bad_rows.append(bad_row)

        except Exception as e:
            signal.alarm(0)
            bad_row = row.copy()
            bad_row["drop_reason"] = str(e)[:300]
            bad_rows.append(bad_row)

    kept_df = pd.DataFrame(kept_rows).reset_index(drop=True)
    bad_df = pd.DataFrame(bad_rows).reset_index(drop=True)

    print(f"{name}: kept {len(kept_df)} / {len(df)}")
    print(f"{name}: dropped {len(bad_df)} unreadable or timeout pairs")

    return kept_df, bad_df

In [17]:

class PairedADNIDataset(Dataset):
    def __init__(self, df, split_name):
        self.df = df.reset_index(drop=True).copy()
        self.split_name = split_name

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx].to_dict()

        dataset = row["dataset"]
        mri_index_key = f"{dataset}_MRI"
        pet_index_key = f"{dataset}_PET"

        mri = get_cached_or_build(row, modality="MRI", index_key=mri_index_key)
        pet = get_cached_or_build(row, modality="PET", index_key=pet_index_key)

        mri = torch.tensor(mri, dtype=torch.float32).unsqueeze(0)
        pet = torch.tensor(pet, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(int(row["label_num"]), dtype=torch.long)

        return {
            "mri": mri,
            "pet": pet,
            "label": label,
            "subject": row["subject"],
            "dataset": row["dataset"],
        }

def make_loader(df, split_name, shuffle):
    ds = PairedADNIDataset(df, split_name=split_name)
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

In [18]:
def make_domain_splits(df, seed=42):
    df = df.reset_index(drop=True).copy()

    train_val_df, test_df = train_test_split(
        df,
        test_size=0.30,
        random_state=seed,
        stratify=df["label_num"],
    )

    val_ratio_within_train = 0.30 / 0.70
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=val_ratio_within_train,
        random_state=seed,
        stratify=train_val_df["label_num"],
    )

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


if PREFILTER_BEFORE_SPLIT:
    print(
        "Filtering readable MRI/PET pairs before splitting so the train/val/test split is made "
        "on the final usable sample set."
    )
    paired_adni1_f, paired_adni1_bad = filter_loadable_pairs(paired_adni1, "paired_adni1")
    paired_adni2_f, paired_adni2_bad = filter_loadable_pairs(paired_adni2, "paired_adni2")
    dropped_all = pd.concat([paired_adni1_bad, paired_adni2_bad], ignore_index=True)
else:
    paired_adni1_f, paired_adni1_bad = paired_adni1.copy(), pd.DataFrame()
    paired_adni2_f, paired_adni2_bad = paired_adni2.copy(), pd.DataFrame()
    dropped_all = pd.DataFrame()

total_usable = len(paired_adni1_f) + len(paired_adni2_f)
print(f"Usable AD/CN pairs in this run: {total_usable}")
print(f"Paper AD/CN total subjects    : {PAPER_AD_CN_TOTAL}")

if total_usable < PAPER_AD_CN_TOTAL:
    print(
        "[warning] Current AD/CN coverage is smaller than the paper's. Matching the paper's accuracy "
        "is unlikely unless the missing subjects and paper-style preprocessing are restored."
    )

if len(paired_adni1_f) < MIN_RELIABLE_SOURCE_SIZE:
    print(
        f"[warning] ADNI1 usable pairs = {len(paired_adni1_f)}. "
        "After the paper-style split, ADNI1->ADNI2 source training will be very data-limited."
    )

adni1_train, adni1_val, adni1_test = make_domain_splits(paired_adni1_f, seed=SEED)
adni2_train, adni2_val, adni2_test = make_domain_splits(paired_adni2_f, seed=SEED)

adni1_train_f, adni1_val_f, adni1_test_f = adni1_train, adni1_val, adni1_test
adni2_train_f, adni2_val_f, adni2_test_f = adni2_train, adni2_val, adni2_test

print("ADNI1 splits:")
for name, dfx in [("train", adni1_train_f), ("val", adni1_val_f), ("test", adni1_test_f)]:
    print(name, dfx.shape, dfx["group"].value_counts().to_dict())

print("\nADNI2 splits:")
for name, dfx in [("train", adni2_train_f), ("val", adni2_val_f), ("test", adni2_test_f)]:
    print(name, dfx.shape, dfx["group"].value_counts().to_dict())


Filtering readable MRI/PET pairs before splitting so the train/val/test split is made on the final usable sample set.


filter paired_adni1:   0%|          | 0/84 [00:00<?, ?it/s]

GDCMSeriesFileNames (0x43cc4a60): No Series were found

ImageSeriesReader (0x43cc4a60): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.0002

ImageSeriesReader (0x43cc4a60): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000398182

ImageSeriesReader (0x43cc4a60): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000297576

GDCMSeriesFileNames (0x43cc4a60): No Series were found

ImageSeriesReader (0x43cc4a60): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000398182

GDCMSeriesFileNames (0x43cc4a60): No Series were found

ImageSeriesReader (0x43cc4a60): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000496364

GDCMSeriesFileNames (0x43cc4a60): No Series were found

GDCMSeriesFileNames (0x43cc4a60): No Series were found

GDCMSeriesFileNames (0x43cc4a60): No Series were found

GDCMSeriesFileNames (0x43cc4a60): No Series were found

ImageSeriesReader (0x43cc4a6

paired_adni1: kept 58 / 84
paired_adni1: dropped 26 unreadable or timeout pairs


ImageSeriesReader (0x44554a70): Non uniform sampling or missing slices detected,  maximum nonuniformity:143.809



filter paired_adni2:   0%|          | 0/281 [00:00<?, ?it/s]

ImageSeriesReader (0x44554a70): Non uniform sampling or missing slices detected,  maximum nonuniformity:177.67

ImageSeriesReader (0x44554a70): Non uniform sampling or missing slices detected,  maximum nonuniformity:177.67

ImageSeriesReader (0x44554a70): Non uniform sampling or missing slices detected,  maximum nonuniformity:177.67

ImageSeriesReader (0x44554a70): Non uniform sampling or missing slices detected,  maximum nonuniformity:177.67

ImageSeriesReader (0x44554a70): Non uniform sampling or missing slices detected,  maximum nonuniformity:177.67

ImageSeriesReader (0x44554a70): Non uniform sampling or missing slices detected,  maximum nonuniformity:177.67

ImageSeriesReader (0x44554a70): Non uniform sampling or missing slices detected,  maximum nonuniformity:177.67

ImageSeriesReader (0x44554a70): Non uniform sampling or missing slices detected,  maximum nonuniformity:177.67

ImageSeriesReader (0x44554a70): Non uniform sampling or missing slices detected,  maximum nonuniformity:

paired_adni2: kept 249 / 281
paired_adni2: dropped 32 unreadable or timeout pairs
Usable AD/CN pairs in this run: 307
Paper AD/CN total subjects    : 459
[warning] Current AD/CN coverage is smaller than the paper's. Matching the paper's accuracy is unlikely unless the missing subjects and paper-style preprocessing are restored.
ADNI1 splits:
train (22, 13) {'CN': 12, 'AD': 10}
val (18, 13) {'AD': 9, 'CN': 9}
test (18, 13) {'CN': 9, 'AD': 9}

ADNI2 splits:
train (99, 13) {'CN': 65, 'AD': 34}
val (75, 13) {'CN': 50, 'AD': 25}
test (75, 13) {'CN': 50, 'AD': 25}


In [19]:
sample_a = adni1_train.iloc[0].to_dict()
mri_a = get_cached_or_build(sample_a, modality="MRI", index_key="ADNI1_MRI")
pet_a = get_cached_or_build(sample_a, modality="PET", index_key="ADNI1_PET")
print("ADNI1 MRI shape:", mri_a.shape, "PET shape:", pet_a.shape)

sample_b = adni2_train.iloc[0].to_dict()
mri_b = get_cached_or_build(sample_b, modality="MRI", index_key="ADNI2_MRI")
pet_b = get_cached_or_build(sample_b, modality="PET", index_key="ADNI2_PET")
print("ADNI2 MRI shape:", mri_b.shape, "PET shape:", pet_b.shape)

ADNI1 MRI shape: (113, 137, 113) PET shape: (113, 137, 113)
ADNI2 MRI shape: (113, 137, 113) PET shape: (113, 137, 113)


In [20]:
print("Usable split summary (filtering was performed before splitting):")
print("ADNI1 usable total:", len(paired_adni1_f))
print("ADNI2 usable total:", len(paired_adni2_f))
print("ADNI1 train/val/test:", len(adni1_train_f), len(adni1_val_f), len(adni1_test_f))
print("ADNI2 train/val/test:", len(adni2_train_f), len(adni2_val_f), len(adni2_test_f))
print("Total dropped before split:", len(dropped_all))

if len(dropped_all) > 0:
    display(dropped_all.head(20))


Usable split summary (filtering was performed before splitting):
ADNI1 usable total: 58
ADNI2 usable total: 249
ADNI1 train/val/test: 22 18 18
ADNI2 train/val/test: 99 75 75
Total dropped before split: 58


,dataset,subject,group,label_num,sex,age,mri_visit,pet_visit,mri_image_id,pet_image_id,mri_date,pet_date,pair_gap_days,drop_reason
0,ADNI1,005_S_0221,AD,1,M,68.0,m06,m06,I25942,I25918,2006-10-06,2006-10-06,0,Could not locate study path for image_id=i2591...
1,ADNI1,005_S_0610,CN,0,M,80.0,m06,m06,I39068,I39087,2007-02-09,2007-02-09,0,Could not locate study path for image_id=i3908...
2,ADNI1,005_S_0929,AD,1,M,83.0,m06,m06,I53858,I57389,2007-05-09,2007-06-19,41,Could not locate study path for image_id=i5738...
3,ADNI1,007_S_0316,AD,1,M,82.0,m06,m06,I28050,I27923,2006-10-30,2006-10-30,0,Could not locate study path for image_id=i2792...
4,ADNI1,009_S_0751,CN,0,M,74.0,m36,m36,I154019,I154497,2009-08-24,2009-09-03,10,Could not locate study path for image_id=i1540...
5,ADNI1,009_S_0842,CN,0,M,77.0,m36,m36,I156154,I156112,2009-10-05,2009-10-06,1,Could not locate study path for image_id=i1561...
6,ADNI1,009_S_0862,CN,0,F,77.0,m36,m36,I157179,I157578,2009-10-14,2009-10-20,6,Could not locate study path for image_id=i1571...
7,ADNI1,029_S_0836,AD,1,M,84.0,m12,m12,I76494,I76321,2007-10-01,2007-09-20,11,All candidate volume loaders failed for /kaggl...
8,ADNI1,029_S_0843,CN,0,M,71.0,sc,bl,I24406,I32461,2006-09-13,2006-11-17,65,All candidate volume loaders failed for /kaggl...
9,ADNI1,029_S_0845,CN,0,M,81.0,m06,m06,I47780,I48324,2007-04-02,2007-04-02,0,All candidate volume loaders failed for /kaggl...


## 5. Improvement 2 model

Architecture:

```mermaid
flowchart LR
    A[MRI Volume] --> B[3D MRI Encoder]
    C[PET Volume] --> D[3D PET Encoder]
    B --> E[MRI Embedding]
    D --> F[PET Embedding]
    E --> G[Gated Fusion]
    F --> G
    E --> H[Absolute Difference]
    F --> H
    H --> G
    G --> I[Fused Patient Embedding]
    I --> J[AD/CN Classifier]
    I --> K[Domain Discriminator via GRL]
    I --> L[Post-hoc Prototype Reliability]
```

The gate is interpretable: a larger gate value means the fused embedding leans more toward MRI; a smaller value means it leans more toward PET.


In [21]:
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None

def grad_reverse(x, lambd=1.0):
    return GradReverse.apply(x, lambd)


class ConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv3d(in_ch, out_ch, kernel_size=3, stride=1, padding=0, bias=False)
        self.bn = nn.BatchNorm3d(out_ch)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class Encoder3D(nn.Module):
    def __init__(self):
        super().__init__()
        channels = [8, 8, 16, 16, 32, 32, 64, 64, 128]
        blocks = []
        in_ch = 1
        for out_ch in channels:
            blocks.append(ConvBlock3D(in_ch, out_ch))
            in_ch = out_ch
        self.blocks = nn.ModuleList(blocks)
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)

    def forward(self, x):
        for i, block in enumerate(self.blocks, start=1):
            x = block(x)
            if i in {2, 4, 6, 8}:
                x = self.pool(x)
        return x


class GatedFusion(nn.Module):
    def __init__(self, in_channels=128, embed_dim=128, dropout=0.5):
        super().__init__()
        self.mri_proj = nn.Sequential(
            nn.Linear(in_channels, embed_dim),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
        )
        self.pet_proj = nn.Sequential(
            nn.Linear(in_channels, embed_dim),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
        )
        self.gate_net = nn.Sequential(
            nn.Linear(embed_dim * 3, embed_dim),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim),
            nn.Sigmoid(),
        )
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, f_mri, f_pet):
        mri_vec = f_mri.mean(dim=(2, 3, 4))
        pet_vec = f_pet.mean(dim=(2, 3, 4))

        mri_emb = self.mri_proj(mri_vec)
        pet_emb = self.pet_proj(pet_vec)

        gate_in = torch.cat([mri_emb, pet_emb, torch.abs(mri_emb - pet_emb)], dim=1)
        gate = self.gate_net(gate_in)

        fused = gate * mri_emb + (1.0 - gate) * pet_emb
        fused = self.norm(fused)
        return fused, mri_emb, pet_emb, gate


class MLPHead(nn.Module):
    def __init__(self, in_dim=128, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.net(x)


class GatedReliabilityMMDDA(nn.Module):
    def __init__(self, dropout=0.5, grl_lambda=0.1):
        super().__init__()
        self.mri_encoder = Encoder3D()
        self.pet_encoder = Encoder3D()
        self.fusion = GatedFusion(in_channels=128, embed_dim=128, dropout=dropout)
        self.classifier = MLPHead(in_dim=128, dropout=dropout)
        self.discriminator = MLPHead(in_dim=128, dropout=dropout)
        self.grl_lambda = grl_lambda

    def encode(self, mri, pet):
        f_mri = self.mri_encoder(mri)
        f_pet = self.pet_encoder(pet)
        fused, mri_emb, pet_emb, gate = self.fusion(f_mri, f_pet)
        return {
            "f_mri": f_mri,
            "f_pet": f_pet,
            "mri_emb": mri_emb,
            "pet_emb": pet_emb,
            "fused": fused,
            "gate": gate,
        }

    def classify(self, fused):
        return self.classifier(fused)

    def discriminate(self, fused):
        x = grad_reverse(fused, self.grl_lambda)
        return self.discriminator(x)


In [22]:


class WeightedFocalLoss(nn.Module):
    def __init__(self, class_weights=None, gamma=2.0):
        super().__init__()
        if class_weights is None:
            self.register_buffer("class_weights", torch.ones(2))
        else:
            self.register_buffer("class_weights", torch.tensor(class_weights, dtype=torch.float32))
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.class_weights.to(logits.device), reduction="none")
        pt = torch.exp(-ce)
        focal = ((1.0 - pt) ** self.gamma) * ce
        return focal.mean()


domain_ce_loss = nn.CrossEntropyLoss()
mse_loss = nn.MSELoss()

def modality_alignment_loss(mri_emb, pet_emb):
    return F.mse_loss(F.normalize(mri_emb, dim=1), F.normalize(pet_emb, dim=1))


def compute_class_weights(df):
    counts = df["label_num"].value_counts().to_dict()
    n0 = counts.get(0, 0)
    n1 = counts.get(1, 0)
    total = max(n0 + n1, 1)
    w0 = total / max(2 * n0, 1)
    w1 = total / max(2 * n1, 1)
    return [float(w0), float(w1)]


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = np.nan

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    spe = tn / (tn + fp + 1e-8)
    sen = tp / (tp + fn + 1e-8)
    bal_acc = 0.5 * (sen + spe)

    return {
        "ACC": acc * 100.0,
        "BAL_ACC": bal_acc * 100.0,
        "SEN": sen * 100.0,
        "SPE": spe * 100.0,
        "AUC": auc * 100.0 if not np.isnan(auc) else np.nan,
        "F1": f1 * 100.0,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "threshold": float(threshold),
    }


def validation_score(metrics, primary=CHECKPOINT_MONITOR):
    primary = str(primary).upper()

    if primary == "AUC" and not np.isnan(metrics.get("AUC", np.nan)):
        return float(metrics["AUC"])
    if primary == "BAL_ACC":
        return float(metrics.get("BAL_ACC", 0.0))
    if primary == "ACC":
        return float(metrics.get("ACC", 0.0))
    if primary == "F1":
        return float(metrics.get("F1", 0.0))

    for fallback_name in ["AUC", "BAL_ACC", "ACC", "F1"]:
        value = metrics.get(fallback_name, np.nan)
        if not np.isnan(value):
            return float(value)

    return float("-inf")


@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    y_true, y_prob = [], []
    gate_means = []

    for batch in loader:
        mri = batch["mri"].to(DEVICE, non_blocking=True)
        pet = batch["pet"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)

        enc = model.encode(mri, pet)
        logits = model.classify(enc["fused"])
        prob = torch.softmax(logits, dim=1)[:, 1]

        y_true.extend(label.cpu().numpy().tolist())
        y_prob.extend(prob.detach().cpu().numpy().tolist())
        gate_means.extend(enc["gate"].detach().mean(dim=1).cpu().numpy().tolist())

    return np.asarray(y_true), np.asarray(y_prob), np.asarray(gate_means)


def tune_threshold(y_true, y_prob, metric="balanced_accuracy"):
    thresholds = np.linspace(0.05, 0.95, 19)
    best_t = 0.5
    best_score = -1.0
    rows = []

    for t in thresholds:
        m = compute_metrics(y_true, y_prob, threshold=t)
        if metric == "f1":
            score = m["F1"]
        else:
            score = m["BAL_ACC"]
        rows.append({"threshold": t, "score": score, "BAL_ACC": m["BAL_ACC"], "F1": m["F1"], "SEN": m["SEN"], "SPE": m["SPE"]})
        if score > best_score:
            best_score = score
            best_t = float(t)

    return best_t, pd.DataFrame(rows)


@torch.no_grad()
def evaluate_model(model, loader, criterion=None, threshold=0.5):
    model.eval()

    y_true = []
    y_prob = []
    total_loss = 0.0
    n = 0
    gate_means = []

    for batch in loader:
        mri = batch["mri"].to(DEVICE, non_blocking=True)
        pet = batch["pet"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)

        enc = model.encode(mri, pet)
        logits = model.classify(enc["fused"])

        if criterion is not None:
            cls = criterion(logits, label)
            align = modality_alignment_loss(enc["mri_emb"], enc["pet_emb"])
            loss = cls + MODALITY_ALIGN_WEIGHT * align
            total_loss += loss.item()
            n += 1

        prob = torch.softmax(logits, dim=1)[:, 1]

        y_true.extend(label.cpu().numpy().tolist())
        y_prob.extend(prob.detach().cpu().numpy().tolist())
        gate_means.extend(enc["gate"].detach().mean(dim=1).cpu().numpy().tolist())

    metrics = compute_metrics(y_true, y_prob, threshold=threshold)
    metrics["loss"] = total_loss / max(n, 1) if criterion is not None else np.nan
    metrics["gate_mean"] = float(np.mean(gate_means)) if len(gate_means) else np.nan
    return metrics


In [23]:
import os
import gc
import torch
import numpy as np

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Real-data GPU test device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Torch:", torch.__version__)
    print("CUDA:", torch.version.cuda)


def check_volume_stats(arr, name):
    arr = np.asarray(arr)
    print(
        name,
        "| shape:", arr.shape,
        "| finite:", np.isfinite(arr).all(),
        "| min:", float(np.nanmin(arr)),
        "| max:", float(np.nanmax(arr)),
        "| mean:", float(np.nanmean(arr)),
        "| std:", float(np.nanstd(arr)),
    )


def gpu_test_real_dataframe(df, split_name, max_items=None):
    print("\n" + "=" * 80)
    print(f"Testing split: {split_name}")
    print("=" * 80)

    model = GatedReliabilityMMDDA(dropout=DROPOUT, grl_lambda=GRL_LAMBDA).to(DEVICE)
    model.train()

    class_weights = compute_class_weights(df)
    criterion = WeightedFocalLoss(class_weights=class_weights, gamma=FOCAL_GAMMA).to(DEVICE)

    if max_items is None:
        max_items = len(df)

    for idx in range(min(len(df), max_items)):
        row = df.iloc[idx].to_dict()
        dataset = row["dataset"]

        print(
            f"\nTesting idx={idx} | dataset={dataset} | subject={row['subject']} | "
            f"MRI={row['mri_image_id']} | PET={row['pet_image_id']}",
            flush=True
        )

        mri = get_cached_or_build(row, modality="MRI", index_key=f"{dataset}_MRI")
        pet = get_cached_or_build(row, modality="PET", index_key=f"{dataset}_PET")

        check_volume_stats(mri, "MRI")
        check_volume_stats(pet, "PET")

        mri_t = torch.tensor(mri, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)
        pet_t = torch.tensor(pet, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)
        label_t = torch.tensor([int(row["label_num"])], dtype=torch.long).to(DEVICE)

        enc = model.encode(mri_t, pet_t)
        logits = model.classify(enc["fused"])

        cls_loss = criterion(logits, label_t)
        align_loss = modality_alignment_loss(enc["mri_emb"], enc["pet_emb"])
        loss = cls_loss + MODALITY_ALIGN_WEIGHT * align_loss

        model.zero_grad(set_to_none=True)
        loss.backward()

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        print(
            "Passed | "
            f"logits={tuple(logits.shape)} | "
            f"fused={tuple(enc['fused'].shape)} | "
            f"gate_mean={float(enc['gate'].detach().mean().cpu()):.4f}"
        )

        del mri_t, pet_t, label_t, enc, logits, loss
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"\nAll tested samples passed for {split_name}.")


def gpu_test_domain_pair(source_df, target_df, source_name, target_name, max_items=5):
    print("\n" + "=" * 80)
    print(f"Testing domain-adaptation path: {source_name} -> {target_name}")
    print("=" * 80)

    model = GatedReliabilityMMDDA(dropout=DROPOUT, grl_lambda=GRL_LAMBDA).to(DEVICE)
    model.train()

    criterion = WeightedFocalLoss(
        class_weights=compute_class_weights(source_df),
        gamma=FOCAL_GAMMA
    ).to(DEVICE)

    n = min(len(source_df), len(target_df), max_items)

    for idx in range(n):
        s_row = source_df.iloc[idx].to_dict()
        t_row = target_df.iloc[idx].to_dict()

        print(
            f"\nTesting pair idx={idx} | "
            f"source_subject={s_row['subject']} | target_subject={t_row['subject']}",
            flush=True
        )

        s_mri = get_cached_or_build(s_row, modality="MRI", index_key=f"{s_row['dataset']}_MRI")
        s_pet = get_cached_or_build(s_row, modality="PET", index_key=f"{s_row['dataset']}_PET")
        t_mri = get_cached_or_build(t_row, modality="MRI", index_key=f"{t_row['dataset']}_MRI")
        t_pet = get_cached_or_build(t_row, modality="PET", index_key=f"{t_row['dataset']}_PET")

        s_mri_t = torch.tensor(s_mri, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)
        s_pet_t = torch.tensor(s_pet, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)
        t_mri_t = torch.tensor(t_mri, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)
        t_pet_t = torch.tensor(t_pet, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)

        s_label_t = torch.tensor([int(s_row["label_num"])], dtype=torch.long).to(DEVICE)

        s_enc = model.encode(s_mri_t, s_pet_t)
        t_enc = model.encode(t_mri_t, t_pet_t)

        cls = criterion(model.classify(s_enc["fused"]), s_label_t)
        align_s = modality_alignment_loss(s_enc["mri_emb"], s_enc["pet_emb"])
        align_t = modality_alignment_loss(t_enc["mri_emb"], t_enc["pet_emb"])

        dom_in = torch.cat([s_enc["fused"], t_enc["fused"]], dim=0)
        dom_logits = model.discriminate(dom_in)
        dom_labels = torch.tensor([1, 0], dtype=torch.long, device=DEVICE)
        dom = domain_ce_loss(dom_logits, dom_labels)

        loss = cls + MODALITY_ALIGN_WEIGHT * (align_s + align_t) + DOMAIN_LOSS_WEIGHT * dom

        model.zero_grad(set_to_none=True)
        loss.backward()

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        print(
            "Passed domain pair | "
            f"source_fused={tuple(s_enc['fused'].shape)} | "
            f"target_fused={tuple(t_enc['fused'].shape)} | "
            f"dom_logits={tuple(dom_logits.shape)}"
        )

        del s_mri_t, s_pet_t, t_mri_t, t_pet_t, s_label_t, s_enc, t_enc, dom_logits, loss
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"\nDomain-adaptation path passed for {source_name} -> {target_name}.")

Real-data GPU test device: cuda
GPU: Tesla T4
Torch: 2.10.0+cu128
CUDA: 12.8


In [24]:
gpu_test_real_dataframe(adni1_train_f, "ADNI1 train")
gpu_test_real_dataframe(adni1_val_f, "ADNI1 val")
gpu_test_real_dataframe(adni1_test_f, "ADNI1 test")

gpu_test_real_dataframe(adni2_train_f, "ADNI2 train")
gpu_test_real_dataframe(adni2_val_f, "ADNI2 val")
gpu_test_real_dataframe(adni2_test_f, "ADNI2 test")

gpu_test_domain_pair(adni1_train_f, adni2_train_f, "ADNI1", "ADNI2", max_items=5)
gpu_test_domain_pair(adni2_train_f, adni1_train_f, "ADNI2", "ADNI1", max_items=5)


Testing split: ADNI1 train

Testing idx=0 | dataset=ADNI1 | subject=005_S_0223 | MRI=I26116 | PET=I26108
MRI | shape: (113, 137, 113) | finite: True | min: -0.8864926099777222 | max: 3.4840657711029053 | mean: -0.1520627737045288 | std: 0.9450310468673706
PET | shape: (113, 137, 113) | finite: True | min: 0.0 | max: 0.9999988675117493 | mean: 0.10699355602264404 | std: 0.22472132742404938
Passed | logits=(1, 2) | fused=(1, 128) | gate_mean=0.5010

Testing idx=1 | dataset=ADNI1 | subject=021_S_0647 | MRI=I38973 | PET=I38959
MRI | shape: (113, 137, 113) | finite: True | min: -0.8331104516983032 | max: 3.0104024410247803 | mean: -0.15231335163116455 | std: 0.9449275732040405
PET | shape: (113, 137, 113) | finite: True | min: 0.0 | max: 0.9999989867210388 | mean: 0.1751057356595993 | std: 0.2360869199037552
Passed | logits=(1, 2) | fused=(1, 128) | gate_mean=0.4982

Testing idx=2 | dataset=ADNI1 | subject=131_S_0123 | MRI=I105091 | PET=I105071
MRI | shape: (113, 137, 113) | finite: True |

In [25]:
def train_stage_1(model, optimizer, source_train_loader, source_val_loader, criterion, epochs, ckpt_path):
    best_score = float("-inf")
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        n = 0

        for batch in source_train_loader:
            mri = batch["mri"].to(DEVICE, non_blocking=True)
            pet = batch["pet"].to(DEVICE, non_blocking=True)
            label = batch["label"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            enc = model.encode(mri, pet)
            logits = model.classify(enc["fused"])

            cls = criterion(logits, label)
            align = modality_alignment_loss(enc["mri_emb"], enc["pet_emb"])
            loss = cls + MODALITY_ALIGN_WEIGHT * align

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            n += 1

        val_metrics = evaluate_model(model, source_val_loader, criterion=criterion, threshold=0.5)
        val_score = validation_score(val_metrics)
        avg_train_loss = train_loss / max(n, 1)

        history.append({
            "epoch": epoch,
            "stage": 1,
            "train_loss": avg_train_loss,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["ACC"],
            "val_bal_acc": val_metrics["BAL_ACC"],
            "val_auc": val_metrics["AUC"],
            "val_f1": val_metrics["F1"],
            "val_gate_mean": val_metrics["gate_mean"],
            "val_score": val_score,
        })

        if val_score > best_score:
            best_score = val_score
            torch.save(model.state_dict(), ckpt_path)

        if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            print(
                f"[Stage 1] epoch {epoch:03d}/{epochs} | "
                f"train_loss={avg_train_loss:.4f} | "
                f"val_acc={val_metrics['ACC']:.2f} | "
                f"val_bal_acc={val_metrics['BAL_ACC']:.2f} | "
                f"val_auc={val_metrics['AUC']:.2f} | "
                f"val_f1={val_metrics['F1']:.2f} | "
                f"gate_mean={val_metrics['gate_mean']:.3f} | "
                f"best_by_{CHECKPOINT_MONITOR.lower()}={best_score:.2f}"
            )

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    return pd.DataFrame(history)


def train_stage_2(model, optimizer, source_train_loader, target_train_loader, source_val_loader, criterion, epochs, ckpt_path):
    best_score = float("-inf")
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        n = 0

        target_iter = cycle(target_train_loader)

        for source_batch in source_train_loader:
            target_batch = next(target_iter)

            s_mri = source_batch["mri"].to(DEVICE, non_blocking=True)
            s_pet = source_batch["pet"].to(DEVICE, non_blocking=True)
            s_label = source_batch["label"].to(DEVICE, non_blocking=True)

            t_mri = target_batch["mri"].to(DEVICE, non_blocking=True)
            t_pet = target_batch["pet"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            s_enc = model.encode(s_mri, s_pet)
            t_enc = model.encode(t_mri, t_pet)

            cls = criterion(model.classify(s_enc["fused"]), s_label)
            align_s = modality_alignment_loss(s_enc["mri_emb"], s_enc["pet_emb"])
            align_t = modality_alignment_loss(t_enc["mri_emb"], t_enc["pet_emb"])

            dom_in = torch.cat([s_enc["fused"], t_enc["fused"]], dim=0)
            dom_logits = model.discriminate(dom_in)
            dom_labels = torch.cat([
                torch.ones(s_enc["fused"].size(0), dtype=torch.long, device=DEVICE),
                torch.zeros(t_enc["fused"].size(0), dtype=torch.long, device=DEVICE),
            ], dim=0)
            dom = domain_ce_loss(dom_logits, dom_labels)

            loss = cls + MODALITY_ALIGN_WEIGHT * (align_s + align_t) + DOMAIN_LOSS_WEIGHT * dom
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n += 1

        val_metrics = evaluate_model(model, source_val_loader, criterion=criterion, threshold=0.5)
        val_score = validation_score(val_metrics)
        avg_epoch_loss = epoch_loss / max(n, 1)

        history.append({
            "epoch": epoch,
            "stage": 2,
            "train_loss": avg_epoch_loss,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["ACC"],
            "val_bal_acc": val_metrics["BAL_ACC"],
            "val_auc": val_metrics["AUC"],
            "val_f1": val_metrics["F1"],
            "val_gate_mean": val_metrics["gate_mean"],
            "val_score": val_score,
        })

        if val_score > best_score:
            best_score = val_score
            torch.save(model.state_dict(), ckpt_path)

        if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            print(
                f"[Stage 2] epoch {epoch:03d}/{epochs} | "
                f"train_loss={avg_epoch_loss:.4f} | "
                f"val_acc={val_metrics['ACC']:.2f} | "
                f"val_bal_acc={val_metrics['BAL_ACC']:.2f} | "
                f"val_auc={val_metrics['AUC']:.2f} | "
                f"val_f1={val_metrics['F1']:.2f} | "
                f"gate_mean={val_metrics['gate_mean']:.3f} | "
                f"best_by_{CHECKPOINT_MONITOR.lower()}={best_score:.2f}"
            )

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    return pd.DataFrame(history)


In [26]:
@torch.no_grad()
def collect_embeddings(model, loader):
    model.eval()
    rows = []
    embeddings = []
    labels = []
    probs = []
    gates = []
    subjects = []
    datasets = []

    for batch in loader:
        mri = batch["mri"].to(DEVICE, non_blocking=True)
        pet = batch["pet"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)

        enc = model.encode(mri, pet)
        logits = model.classify(enc["fused"])
        prob = torch.softmax(logits, dim=1)[:, 1]

        embeddings.append(enc["fused"].detach().cpu())
        labels.extend(label.cpu().numpy().tolist())
        probs.extend(prob.detach().cpu().numpy().tolist())
        gates.extend(enc["gate"].detach().mean(dim=1).cpu().numpy().tolist())
        subjects.extend(batch["subject"])
        datasets.extend(batch["dataset"])

    if len(embeddings) == 0:
        return None

    emb = torch.cat(embeddings, dim=0).numpy()
    labels = np.asarray(labels).astype(int)
    probs = np.asarray(probs).astype(float)
    gates = np.asarray(gates).astype(float)

    return {
        "emb": emb,
        "labels": labels,
        "probs": probs,
        "gates": gates,
        "subjects": subjects,
        "datasets": datasets,
    }


def fit_source_prototypes(model, source_train_loader):
    data = collect_embeddings(model, source_train_loader)
    emb = data["emb"]
    labels = data["labels"]

    prototypes = {}
    for cls in [0, 1]:
        mask = labels == cls
        if mask.sum() == 0:
            raise RuntimeError(f"No source samples for class {cls}; cannot build prototype.")
        prototypes[cls] = emb[mask].mean(axis=0)

    return prototypes


def prototype_reliability_dataframe(model, loader, prototypes, split_name, threshold=0.5):
    data = collect_embeddings(model, loader)
    emb = data["emb"]
    labels = data["labels"]
    probs = data["probs"]
    gates = data["gates"]

    p0 = prototypes[0]
    p1 = prototypes[1]

    d0 = np.linalg.norm(emb - p0[None, :], axis=1)
    d1 = np.linalg.norm(emb - p1[None, :], axis=1)
    nearest_dist = np.minimum(d0, d1)
    proto_pred = (d1 < d0).astype(int)

    pred = (probs >= threshold).astype(int)
    confidence = np.maximum(probs, 1.0 - probs)
    correct = (pred == labels).astype(int)

    df = pd.DataFrame({
        "split": split_name,
        "subject": data["subjects"],
        "dataset": data["datasets"],
        "label": labels,
        "prob_AD": probs,
        "pred": pred,
        "correct": correct,
        "confidence": confidence,
        "gate_mean": gates,
        "dist_to_CN_proto": d0,
        "dist_to_AD_proto": d1,
        "nearest_proto_distance": nearest_dist,
        "prototype_pred": proto_pred,
        "prototype_margin_abs": np.abs(d0 - d1),
    })
    return df


def summarize_reliability(df):
    out = {
        "n": len(df),
        "mean_confidence": df["confidence"].mean(),
        "mean_gate": df["gate_mean"].mean(),
        "mean_nearest_proto_distance": df["nearest_proto_distance"].mean(),
        "accuracy_all": df["correct"].mean() * 100.0,
    }

    if len(df) >= 4:
        conf_cut = df["confidence"].median()
        near_cut = df["nearest_proto_distance"].median()

        high_conf = df[df["confidence"] >= conf_cut]
        low_conf = df[df["confidence"] < conf_cut]
        near_proto = df[df["nearest_proto_distance"] <= near_cut]
        far_proto = df[df["nearest_proto_distance"] > near_cut]

        out["high_conf_acc"] = high_conf["correct"].mean() * 100.0 if len(high_conf) else np.nan
        out["low_conf_acc"] = low_conf["correct"].mean() * 100.0 if len(low_conf) else np.nan
        out["near_proto_acc"] = near_proto["correct"].mean() * 100.0 if len(near_proto) else np.nan
        out["far_proto_acc"] = far_proto["correct"].mean() * 100.0 if len(far_proto) else np.nan

        correct_df = df[df["correct"] == 1]
        wrong_df = df[df["correct"] == 0]
        out["correct_mean_proto_dist"] = correct_df["nearest_proto_distance"].mean() if len(correct_df) else np.nan
        out["wrong_mean_proto_dist"] = wrong_df["nearest_proto_distance"].mean() if len(wrong_df) else np.nan

    return out


In [27]:

def run_transfer_experiment(source_name, target_name, source_train_df, source_val_df, source_test_df,
                            target_train_df, target_val_df, target_test_df):
    print("=" * 80)
    print(f"Running Improvement 2 transfer experiment: {source_name} -> {target_name}")
    print("=" * 80)
    print(
        f"source train/val/test = {len(source_train_df)}/{len(source_val_df)}/{len(source_test_df)} | "
        f"target train/val/test = {len(target_train_df)}/{len(target_val_df)}/{len(target_test_df)}"
    )

    source_train_loader = make_loader(source_train_df, f"{source_name}_train", shuffle=True)
    source_val_loader   = make_loader(source_val_df,   f"{source_name}_val",   shuffle=False)
    source_test_loader  = make_loader(source_test_df,  f"{source_name}_test",  shuffle=False)

    target_train_loader = make_loader(target_train_df, f"{target_name}_train", shuffle=True)
    target_val_loader   = make_loader(target_val_df,   f"{target_name}_val",   shuffle=False)
    target_test_loader  = make_loader(target_test_df,  f"{target_name}_test",  shuffle=False)

    class_weights = compute_class_weights(source_train_df)
    print("Source-train class weights [CN, AD]:", class_weights)

    criterion = WeightedFocalLoss(class_weights=class_weights, gamma=FOCAL_GAMMA).to(DEVICE)
    model = GatedReliabilityMMDDA(dropout=DROPOUT, grl_lambda=GRL_LAMBDA).to(DEVICE)

    exp_key = f"{source_name}_to_{target_name}"

    stage1_ckpt = CHECKPOINT_ROOT / f"{exp_key}__stage1_best.pt"
    stage2_ckpt = CHECKPOINT_ROOT / f"{exp_key}__stage2_best.pt"

    optimizer_stage1 = torch.optim.Adam(model.parameters(), lr=LR)
    hist1 = train_stage_1(
        model=model,
        optimizer=optimizer_stage1,
        source_train_loader=source_train_loader,
        source_val_loader=source_val_loader,
        criterion=criterion,
        epochs=PRETRAIN_EPOCHS,
        ckpt_path=stage1_ckpt,
    )

    optimizer_stage2 = torch.optim.Adam(model.parameters(), lr=LR)
    hist2 = train_stage_2(
        model=model,
        optimizer=optimizer_stage2,
        source_train_loader=source_train_loader,
        target_train_loader=target_train_loader,
        source_val_loader=source_val_loader,
        criterion=criterion,
        epochs=ADAPT_EPOCHS,
        ckpt_path=stage2_ckpt,
    )

    y_val, p_val, _ = collect_predictions(model, source_val_loader)
    tuned_threshold, threshold_table = tune_threshold(y_val, p_val, metric=THRESHOLD_METRIC)
    print(f"Tuned threshold from source validation ({THRESHOLD_METRIC}): {tuned_threshold:.2f}")

    source_test_metrics = evaluate_model(model, source_test_loader, criterion=criterion, threshold=tuned_threshold)
    target_val_metrics = evaluate_model(model, target_val_loader, criterion=criterion, threshold=tuned_threshold)
    target_test_metrics = evaluate_model(model, target_test_loader, criterion=criterion, threshold=tuned_threshold)

    prototypes = fit_source_prototypes(model, source_train_loader)
    rel_source_test = prototype_reliability_dataframe(model, source_test_loader, prototypes, f"{exp_key}_source_test", threshold=tuned_threshold)
    rel_target_val = prototype_reliability_dataframe(model, target_val_loader, prototypes, f"{exp_key}_target_val", threshold=tuned_threshold)
    rel_target_test = prototype_reliability_dataframe(model, target_test_loader, prototypes, f"{exp_key}_target_test", threshold=tuned_threshold)

    reliability_summary = pd.DataFrame([
        {"experiment": exp_key, "split": "source_test", **summarize_reliability(rel_source_test)},
        {"experiment": exp_key, "split": "target_val", **summarize_reliability(rel_target_val)},
        {"experiment": exp_key, "split": "target_test", **summarize_reliability(rel_target_test)},
    ])

    result = {
        "experiment": exp_key,
        "threshold": tuned_threshold,
        "threshold_table": threshold_table,
        "source_test": source_test_metrics,
        "target_val": target_val_metrics,
        "target_test": target_test_metrics,
        "history_stage1": hist1,
        "history_stage2": hist2,
        "reliability_source_test": rel_source_test,
        "reliability_target_val": rel_target_val,
        "reliability_target_test": rel_target_test,
        "reliability_summary": reliability_summary,
    }

    print("\nTarget test metrics:")
    for k in ["ACC", "BAL_ACC", "SEN", "SPE", "AUC", "F1"]:
        print(f"{k}: {target_test_metrics[k]:.2f}")
    print("Threshold:", tuned_threshold)
    print("\nReliability summary:")
    display(reliability_summary)

    return result


## 6. Train both transfer directions

This uses the same split objects as the baseline:
- ADNI1 → ADNI2
- ADNI2 → ADNI1



In [28]:

result_12 = run_transfer_experiment(
    source_name="ADNI1",
    target_name="ADNI2",
    source_train_df=adni1_train_f,
    source_val_df=adni1_val_f,
    source_test_df=adni1_test_f,
    target_train_df=adni2_train_f,
    target_val_df=adni2_val_f,
    target_test_df=adni2_test_f,
)

result_21 = run_transfer_experiment(
    source_name="ADNI2",
    target_name="ADNI1",
    source_train_df=adni2_train_f,
    source_val_df=adni2_val_f,
    source_test_df=adni2_test_f,
    target_train_df=adni1_train_f,
    target_val_df=adni1_val_f,
    target_test_df=adni1_test_f,
)


Running Improvement 2 transfer experiment: ADNI1 -> ADNI2
source train/val/test = 22/18/18 | target train/val/test = 99/75/75
Source-train class weights [CN, AD]: [0.9166666666666666, 1.1]
[Stage 1] epoch 001/150 | train_loss=0.1987 | val_acc=50.00 | val_bal_acc=50.00 | val_auc=40.74 | val_f1=0.00 | gate_mean=0.499 | best_by_auc=40.74
[Stage 1] epoch 010/150 | train_loss=0.1567 | val_acc=66.67 | val_bal_acc=66.67 | val_auc=80.25 | val_f1=66.67 | gate_mean=0.501 | best_by_auc=81.48
[Stage 1] epoch 020/150 | train_loss=0.1856 | val_acc=66.67 | val_bal_acc=66.67 | val_auc=77.78 | val_f1=66.67 | gate_mean=0.500 | best_by_auc=81.48
[Stage 1] epoch 030/150 | train_loss=0.1540 | val_acc=66.67 | val_bal_acc=66.67 | val_auc=77.78 | val_f1=70.00 | gate_mean=0.500 | best_by_auc=81.48
[Stage 1] epoch 040/150 | train_loss=0.1881 | val_acc=66.67 | val_bal_acc=66.67 | val_auc=77.78 | val_f1=70.00 | gate_mean=0.501 | best_by_auc=82.72
[Stage 1] epoch 050/150 | train_loss=0.1803 | val_acc=72.22 | val_b

,experiment,split,n,mean_confidence,mean_gate,mean_nearest_proto_distance,accuracy_all,high_conf_acc,low_conf_acc,near_proto_acc,far_proto_acc,correct_mean_proto_dist,wrong_mean_proto_dist
0,ADNI1_to_ADNI2,source_test,18,0.523266,0.500539,9.300358,55.555556,55.555556,55.555556,55.555556,55.555556,9.206337,9.417885
1,ADNI1_to_ADNI2,target_val,75,0.519768,0.500711,10.695127,66.666667,57.894737,75.675676,65.789474,67.567568,10.739615,10.606153
2,ADNI1_to_ADNI2,target_test,75,0.521561,0.500450,10.530139,66.666667,65.789474,67.567568,63.157895,70.270270,10.597452,10.395511


Running Improvement 2 transfer experiment: ADNI2 -> ADNI1
source train/val/test = 99/75/75 | target train/val/test = 22/18/18
Source-train class weights [CN, AD]: [0.7615384615384615, 1.4558823529411764]
[Stage 1] epoch 001/150 | train_loss=0.2160 | val_acc=61.33 | val_bal_acc=64.00 | val_auc=64.24 | val_f1=55.38 | gate_mean=0.501 | best_by_auc=64.24
[Stage 1] epoch 010/150 | train_loss=0.2107 | val_acc=56.00 | val_bal_acc=60.00 | val_auc=55.92 | val_f1=52.17 | gate_mean=0.502 | best_by_auc=64.24
[Stage 1] epoch 020/150 | train_loss=0.2062 | val_acc=41.33 | val_bal_acc=50.00 | val_auc=56.48 | val_f1=46.34 | gate_mean=0.501 | best_by_auc=64.24
[Stage 1] epoch 030/150 | train_loss=0.2048 | val_acc=48.00 | val_bal_acc=58.00 | val_auc=52.32 | val_f1=53.01 | gate_mean=0.502 | best_by_auc=64.24
[Stage 1] epoch 040/150 | train_loss=0.2015 | val_acc=37.33 | val_bal_acc=50.00 | val_auc=56.00 | val_f1=48.35 | gate_mean=0.501 | best_by_auc=64.24
[Stage 1] epoch 050/150 | train_loss=0.1969 | val_a

,experiment,split,n,mean_confidence,mean_gate,mean_nearest_proto_distance,accuracy_all,high_conf_acc,low_conf_acc,near_proto_acc,far_proto_acc,correct_mean_proto_dist,wrong_mean_proto_dist
0,ADNI2_to_ADNI1,source_test,75,0.515368,0.501049,9.022614,42.666667,36.842105,48.648649,42.105263,43.243243,9.015364,9.028011
1,ADNI2_to_ADNI1,target_val,18,0.516816,0.501653,10.864421,55.555556,55.555556,55.555556,55.555556,55.555556,10.941262,10.768370
2,ADNI2_to_ADNI1,target_test,18,0.513263,0.501352,10.499882,50.000000,66.666667,33.333333,66.666667,33.333333,10.210996,10.788769
